In [1]:
!python --version

Python 3.14.5


The system cannot find the path specified.


## ___World Checklist of Vascular Plants (WCVP) homotypic synonyms___
----------------

In [2]:
from collections import namedtuple

import pandas as pd
import numpy as np

In [3]:
# look up https://diatoms.org/news/faq-what-are-homotypic-and-heterotypic-synonyms for homotypic synonyms vs heterotypic synonyms

In [4]:
# World Checklist of Vascular Plants (WCVP) dataset from => https://sftp.kew.org/pub/data-repositories/WCVP/

wcvp = pd.read_csv(r"../../data/chapter2/wcvp/wcvp_names.csv", low_memory=False, sep='|', true_values=['T'])

WCVP_COLUMNS = [
    "taxon_name", # full binominal name
    "plant_name_id", # World Checklist of Vascular Plants (WCVP) identifier
    "accepted_plant_name_id", # the ID of the accepted name of this taxon. Where the taxon_status is "Accepted", this will be identical to the plant_name_id value. May be empty if taxon status is unplaced, ilegitimate, or in some cases where the accepted name is not a vascular plant (e.g. a moss, alga or animal).
    "basionym_plant_name_id", # ID of the original name that taxon_name was derived from. If there is a parenthetical author it is a basionym. If there is a replaced synonym author it is a replaced synonym. If empty there have been no name changes. 
    "taxon_status", # indication of nomenclatural status and taxonomic opinion regarding the name
    "homotypic_synonym" # boolan indicating if the name is a homotypic synonym
]

In [5]:
# only pick the records that are homotypic synonyms; i.e. (taxon_status=="Synonym") and (homotypic_synonym==True)

synonym_pairs = wcvp.loc[:, WCVP_COLUMNS].query(r"taxon_status=='Synonym' and homotypic_synonym and (plant_name_id!=accepted_plant_name_id)").loc[:, ("taxon_name", "plant_name_id", "accepted_plant_name_id")]
synonym_pairs = synonym_pairs.astype({"plant_name_id": int, "accepted_plant_name_id": int}).reset_index(drop=True)

In [6]:
# replace the WCVP identifiers with binominal names

wcvp_synonyms_lookup_table = pd.merge(left=synonym_pairs, left_on="accepted_plant_name_id", right=wcvp.loc[:, WCVP_COLUMNS], right_on="plant_name_id", how="left", suffixes=("_syn", '')).loc[:, ("taxon_name", "taxon_name_syn")]
wcvp_synonyms_lookup_table

,taxon_name,taxon_name_syn
0,Caladenia minorata,Caladenia glossodia
1,Chiloglottis gunnii,Caladenia gunnii
2,Volkameria acerbiana,Clerodendrum acerbianum
3,Veronica sibthorpioides,Cochlidiosperma sibthorpioides
4,Veronica sibthorpioides,Veronica hederifolia subsp. sibthorpioides
...,...,...
281047,Wedelia subpetiolata,Aspilia subpetiolata
281048,Wedelia tomentosa,Aspilia tomentosa
281049,Wedelia trichostephia,Seruneum trichostephia
281050,Wedelia trichostephia,Trichostemma hispidum


In [7]:
wcvp_synonyms_lookup_table.rename({"taxon_name": "name", "taxon_name_syn": "synonym"}, axis=1)

,name,synonym
0,Caladenia minorata,Caladenia glossodia
1,Chiloglottis gunnii,Caladenia gunnii
2,Volkameria acerbiana,Clerodendrum acerbianum
3,Veronica sibthorpioides,Cochlidiosperma sibthorpioides
4,Veronica sibthorpioides,Veronica hederifolia subsp. sibthorpioides
...,...,...
281047,Wedelia subpetiolata,Aspilia subpetiolata
281048,Wedelia tomentosa,Aspilia tomentosa
281049,Wedelia trichostephia,Seruneum trichostephia
281050,Wedelia trichostephia,Trichostemma hispidum


In [8]:
wcvp_synonyms_lookup_table.rename({"taxon_name": "name", "taxon_name_syn": "synonym"}, axis=1).to_csv(r"../../data/chapter2/wcvp/synonyms.csv", index=False)

In [9]:
synonym = namedtuple(typename="synonym", field_names=["name", "synonym"])

In [10]:
# redo the synonym extraction for the final subset - some species are mising synonyms even though the WCVP dataset has synonyms for them

final = pd.read_csv(r"../../data/chapter2/FRED/subsets/final.csv")
names = pd.read_csv(r"../../data/chapter2/FRED/subsets/final.csv").binominal

In [11]:
wcvp_synonyms_lookup_table.query(r"taxon_name.isin(@names)")

,taxon_name,taxon_name_syn
116,Metrosideros umbellata,Agalmanthus umbellata
264,Inga ruiziana,Feuilleea ruiziana
479,Heptapleurum heptaphyllum,Aralia heptaphylla
603,Galium aparine,Asterophyllum aparine
693,Syzygium acuminatissimum,Acmena acuminatissima
...,...,...
280952,Pappobolus microphyllus,Helianthus microphyllus
280966,Helianthus praecox,Helianthus debilis subsp. praecox
280967,Helianthus praecox,Helianthus cucumerifolius var. praecox
280969,Helianthus radula,Helianthus atrorubens subsp. radula


In [12]:
# binominal names in FRED 4.0 in taxon_name column of the synonym table
synonyms = [synonym(name=name, synonym=tuple(df.taxon_name_syn.values)) for (name, df) in wcvp_synonyms_lookup_table.query(r"taxon_name.isin(@names)").groupby("taxon_name", as_index=False)]

In [13]:
wcvp_synonyms_lookup_table.query(r"taxon_name_syn.isin(@names)")

,taxon_name,taxon_name_syn
2485,Ragala sanguinolenta,Chrysophyllum sanguinolentum
30916,Lycopodium volubile,Pseudodiphasium volubile
50455,Blechnum discolor,Lomaria discolor
50654,Onoclea struthiopteris,Matteuccia struthiopteris
53136,Camphora officinarum,Cinnamomum camphora
53148,Camphora micrantha,Cinnamomum micranthum
53152,Camphora parthenoxylon,Cinnamomum parthenoxylon
65897,Blechnum novae-zelandiae,Parablechnum novae-zelandiae
75193,Camphora glandulifera,Cinnamomum glanduliferum
80586,Blechnum procerum,Parablechnum procerum


In [14]:
# binominal names in FRED 4.0 in taxon_name_syn column of the synonym table
synonyms += [synonym(name=name, synonym=tuple(df.taxon_name.values)) for (name, df) in wcvp_synonyms_lookup_table.query(r"taxon_name_syn.isin(@names)").groupby("taxon_name_syn", as_index=False)]

In [18]:
len(synonyms) # total unique species with synonyms 

848

In [19]:
synonyms = pd.DataFrame(synonyms).sort_values("name")
synonyms

,name,synonym
0,Abies alba,"(Pinus alba, Pinus abies var. pectinata, Pinus..."
1,Abies nephrolepis,"(Abies veitchii var. nephrolepis, Abies sibiri..."
2,Acacia auriculiformis,"(Racosperma auriculiforme,)"
3,Acacia crassicarpa,"(Racosperma crassicarpum,)"
4,Acacia mangium,"(Racosperma mangium,)"
...,...,...
821,Wurfbainia villosa,"(Cardamomum villosum, Elettaria villosa, Zingi..."
822,Xanthisma spinulosum,"(Dieteria spinulosa, Sideranthus spinulosus, H..."
823,Zabelia biflora,"(Abelia biflora,)"
824,Zizia aurea,"(Smyrnium aureum, Sison aureum, Carum aureum, ..."


In [20]:
synonyms.to_csv(r"../../data/chapter2/FRED/subsets/synonyms.csv", index=False)

In [23]:
# the species.xlsx file needs some inteventions

In [27]:
pd.merge(left=pd.read_excel(r"../../data/chapter2/FRED/subsets/species.xlsx", sheet_name="combined"), left_on="binominal", right=synonyms, right_on="name", how="left").drop(["name", "synonyms"], axis=1)

,binominal,F01286,F01289,F01290_not_crosschecked,reconciled_state,source,state_revision_1,revision_1_source,state_revision_2,revision_2_source,synonym
0,Abies alba,Abies,Pinaceae,Pinales,EcM,FRED4,EcM,NaN,EcM,NaN,"(Pinus alba, Pinus abies var. pectinata, Pinus..."
1,Abies nephrolepis,Abies,Pinaceae,Pinales,EcM,FRED4,EcM,NaN,EcM,NaN,"(Abies veitchii var. nephrolepis, Abies sibiri..."
2,Acacia auriculiformis,Acacia,Fabaceae,Fabales,AM,FRED4,AM,NaN,EcMAM,"(Soudzilovskaia et al., 2020)","(Racosperma auriculiforme,)"
3,Acacia crassicarpa,Acacia,Fabaceae,Fabales,EcMAM,FRED4,EcMAM,NaN,EcMAM,NaN,"(Racosperma crassicarpum,)"
4,Acacia mangium,Acacia,Fabaceae,Fabales,EcMAM,FRED4,EcMAM,NaN,EcMAM,NaN,"(Racosperma mangium,)"
...,...,...,...,...,...,...,...,...,...,...,...
1296,Xanthophyllum tenue,Xanthophyllum,Polygalaceae,Fabales,AM,FRED4,AM,NaN,AM,NaN,NaN
1297,Zabelia biflora,Zabelia,Caprifoliaceae,Dipsacales,AM,FRED4,AM,NaN,AM,NaN,"(Abelia biflora,)"
1298,Zamia lucayana,Zamia,Zamiaceae,Cycadales,AM,FRED4,AM,NaN,AM,NaN,NaN
1299,Zizia aurea,Zizia,Apiaceae,Apiales,AM,FungalRoot,AM,NaN,AM,NaN,"(Smyrnium aureum, Sison aureum, Carum aureum, ..."
